In [44]:
import sys
sys.path.append("..")
import pandas as pd
from src.cleaning import clean_patients, clean_encounters

patients = clean_patients(pd.read_csv("../data/raw/patients.csv"))

encounters = clean_encounters(pd.read_csv("../data/raw/encounters.csv"))

# a clinical state/diagnosis associated with the patient.
conditions = pd.read_csv("../data/raw/conditions.csv", dtype={"CODE": str})
conditions.shape
conditions.columns
conditions.info()
conditions.isna().sum()
conditions.head(10)



<class 'pandas.DataFrame'>
RangeIndex: 3784 entries, 0 to 3783
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   START        3784 non-null   str  
 1   STOP         2818 non-null   str  
 2   PATIENT      3784 non-null   str  
 3   ENCOUNTER    3784 non-null   str  
 4   SYSTEM       3784 non-null   str  
 5   CODE         3784 non-null   str  
 6   DESCRIPTION  3784 non-null   str  
dtypes: str(7)
memory usage: 207.1 KB


,START,STOP,PATIENT,ENCOUNTER,SYSTEM,CODE,DESCRIPTION
0,1997-04-22,NaN,a1733070-046a-4506-bba6-47f32652e9d7,a1733070-046a-4506-13b1-f518d757cdd0,http://snomed.info/sct,224299000,Received higher education (finding)
1,1997-04-22,NaN,a1733070-046a-4506-bba6-47f32652e9d7,a1733070-046a-4506-13b1-f518d757cdd0,http://snomed.info/sct,713458007,Lack of access to transportation (finding)
2,1989-01-10,NaN,f9ba83f7-9940-16ae-0854-24bd34bf1843,f9ba83f7-9940-16ae-158c-5cd0e5a140b9,http://snomed.info/sct,160968000,Risk activity involvement (finding)
3,2013-05-14,NaN,a1733070-046a-4506-bba6-47f32652e9d7,a1733070-046a-4506-0d73-cf2822965f28,http://snomed.info/sct,266934004,Transport problem (finding)
4,2014-02-06,NaN,a1733070-046a-4506-bba6-47f32652e9d7,a1733070-046a-4506-175b-fc14062cdb07,http://snomed.info/sct,82423001,Chronic pain (finding)
5,2014-02-06,NaN,a1733070-046a-4506-bba6-47f32652e9d7,a1733070-046a-4506-175b-fc14062cdb07,http://snomed.info/sct,278860009,Chronic low back pain (finding)
6,2014-02-06,NaN,a1733070-046a-4506-bba6-47f32652e9d7,a1733070-046a-4506-175b-fc14062cdb07,http://snomed.info/sct,1121000119107,Chronic neck pain (finding)
7,1990-01-16,NaN,f9ba83f7-9940-16ae-0854-24bd34bf1843,f9ba83f7-9940-16ae-665a-6f37a1042b1a,http://snomed.info/sct,105531004,Housing unsatisfactory (finding)
8,1990-01-16,NaN,f9ba83f7-9940-16ae-0854-24bd34bf1843,f9ba83f7-9940-16ae-665a-6f37a1042b1a,http://snomed.info/sct,224299000,Received higher education (finding)
9,2003-02-04,NaN,f9ba83f7-9940-16ae-0854-24bd34bf1843,f9ba83f7-9940-16ae-66d7-aab90f06f82e,http://snomed.info/sct,87433001,Pulmonary emphysema (disorder)


In [45]:
# Patient-level vs record-level analysis

total_patients = patients['Id'].nunique()
patients_with_conditions = conditions['PATIENT'].nunique()

pct_patients_with_conditions = (patients_with_conditions / total_patients) * 100

print(f"Percentage of patients with at least one condition: {pct_patients_with_conditions:.2f}%")


Percentage of patients with at least one condition: 100.00%


In [46]:
# what are the most common conditions ?

condition_counts = conditions["DESCRIPTION"].value_counts().head(10)
condition_percentage = (condition_counts / len(conditions) * 100).round(2)

topconditions =  pd.DataFrame({
    'Count' : condition_counts,
    'Percentage' : condition_percentage
})
print(topconditions)


                                   Count  Percentage
DESCRIPTION                                         
Medication review due (situation)    784       20.72
Stress (finding)                     285        7.53
Full-time employment (finding)       275        7.27
Gingivitis (disorder)                270        7.14
Part-time employment (finding)       175        4.62
Social isolation (finding)           107        2.83
Limited social contact (finding)     106        2.80
Viral sinusitis (disorder)           103        2.72
Gingival disease (disorder)           85        2.25
Not in labor force (finding)          80        2.11


In [47]:
# categorized number of conditions 
conditions["category"] = (
    conditions["DESCRIPTION"].str.extract(r"\(([^)]+)\)$")[0].fillna("untagged")
)
conditions["category"].value_counts()



category
finding                    1709
disorder                   1202
situation                   842
morphologic abnormality      16
untagged                      8
person                        7
Name: count, dtype: int64

In [48]:
# Diagnosis codes
conditions[["CODE", "DESCRIPTION"]].head(20)

,CODE,DESCRIPTION
0,224299000,Received higher education (finding)
1,713458007,Lack of access to transportation (finding)
2,160968000,Risk activity involvement (finding)
3,266934004,Transport problem (finding)
4,82423001,Chronic pain (finding)
5,278860009,Chronic low back pain (finding)
6,1121000119107,Chronic neck pain (finding)
7,105531004,Housing unsatisfactory (finding)
8,224299000,Received higher education (finding)
9,87433001,Pulmonary emphysema (disorder)


In [49]:
# conditions vs patients demographics
patients_with_conditions = conditions.groupby("PATIENT").size().sort_values(ascending=False)
patients_with_conditions.describe()

count    108.000000
mean      35.037037
std       38.892838
min        2.000000
25%       18.000000
50%       30.000000
75%       38.250000
max      319.000000
dtype: float64

In [50]:
# How many conditions does each patient have?
conditions_per_patient = (
    conditions.groupby("PATIENT")
    .size()
    .sort_values(ascending=False)
)
print(conditions_per_patient.head(10))

PATIENT
5d84e6a3-b4bd-63d6-57c5-cada0916490d    319
7ad140ab-bfae-c3ae-20a3-2244b1c4d0e2    219
688c8453-ba6f-7dec-03c3-eaa27d6df1a4    128
01a006ce-6457-50c2-8e0a-fb58fc310a86    109
8d7f6a31-31ba-da9c-2b57-03ee0f7577a0    103
c1e9a9fb-45ec-4ee4-c946-4ffc5dfa93ea     93
53e4891a-9108-67ef-d973-3b6a98404249     78
ba234ff2-cefe-dfee-935a-8ea2378da8c2     57
885c1eeb-1b4c-8a3a-bfc7-a68897e470a6     55
2eb14889-46f2-06de-4f6e-fea5634e0d85     51
dtype: int64
